# PRISM: upload the releasable embeddings to the Hugging Face Hub

Runs on Colab with Google Drive mounted. **Run `01_embedding_inventory.ipynb` first.**
It writes `MyDrive/PRISM/embedding_inventory.csv`.

Steps in this notebook:

1. Read `embedding_inventory.csv` and apply the release policy (`RELEASE_POLICY.md`). Released: models CLIP, PLIP, GigaPath, H-Optimus-0, MIDNIGHT × datasets PCam, CRC, BRACS, LungHist700, SPIDER-Breast and Camelyon17. Withheld: MHIST (dataset licence) and UNI, CONCH, VIRCHOW2 (model licence).
2. Re-compute the SHA256 of every file to be released and stop on any mismatch with the inventory.
3. Compute numerical fingerprints for **all** feature files and write `checksums.csv` covering every inventory file, released and withheld.
4. Stage the release in `/content/hf_staging`: symlinks to the `.npy` files on Drive, parquet copies for the `datasets` configs, the dataset card, the policy, code and splits.
5. Create the repo as **private**, enable gated access, upload with `upload_large_folder` (resumable: if the session dies, re-run all cells), check the remote file list, and print the URL.

The repo is never made public automatically. The last cell does that and is commented out.

**Before running**

- Copy the folder `cam-ready/embeddings_release/` (README.md, RELEASE_POLICY.md, `code/`, `splits/`) to `MyDrive/PRISM/embeddings_release/`.
- Optional but recommended: run `code/make_split_indices.py` (cell 3b) to write `splits/<dataset>.json` for the six main datasets.
- Add a Colab secret `HF_TOKEN` (a token with **write** scope) and enable notebook access to it.
- Set `REPO_ID` below.

In [ ]:
!pip -q install -U "huggingface_hub>=0.27" pyarrow
from google.colab import drive, userdata
drive.mount('/content/drive')

In [ ]:
import os, sys, json, shutil, hashlib, re, time
import numpy as np, pandas as pd

# ---------------- configuration ----------------
ROOT     = '/content/drive/MyDrive/PRISM'            # contains embeddings/, embeddings_camelyon17/, embedding_inventory.csv
REL_DIR  = f'{ROOT}/embeddings_release'              # copy of cam-ready/embeddings_release
STAGE    = '/content/hf_staging'                     # local staging folder (symlinks + parquet)
REPO_ID  = 'YOUR_HF_USERNAME/PRISM-embeddings'       # <-- set this
GATED    = 'auto'          # 'auto' (click-through, recommended), 'manual', or False
MAKE_PARQUET  = True       # parquet copies for the datasets configs (about doubles the upload size)
COMPUTE_STATS = True       # fingerprints for checksums.csv (needed by extract_embeddings.py verify)
REVERIFY_WITHHELD = False  # also re-hash withheld files (otherwise their sha256 is taken from the inventory)
DRY_RUN  = False           # True: build staging and checksums.csv but do not upload

# ---------------- release policy (RELEASE_POLICY.md, sections 1-3) ----------------
MODELS   = ['clip', 'plip', 'conch', 'uni', 'virchow2', 'gigapath', 'h_optimus_0', 'midnight']
DATASETS = ['pcam', 'crc', 'mhist', 'bracs', 'lunghist700', 'spider_breast']
RELEASE_MODELS   = {'clip', 'plip', 'gigapath', 'h_optimus_0', 'midnight'}
RELEASE_DATASETS = {'pcam', 'crc', 'bracs', 'lunghist700', 'spider_breast', 'camelyon17'}
WITHHELD_MODEL_REASON = {
    'uni':      'model licence: UNI CC-BY-NC-ND 4.0 + gated terms (datasets created from the model are derivatives)',
    'conch':    'model licence: CONCH CC-BY-NC-ND 4.0 + gated terms (datasets created from the model are derivatives)',
    'virchow2': 'model licence: Virchow2 CC-BY-NC-ND 4.0 + gated terms (datasets created from the model are derivatives)',
}
WITHHELD_DATASET_REASON = {'mhist': 'dataset licence: MHIST Research Use Agreement s.4 (no redistribution) and s.6 (no derivative works)'}
DATASET_LICENSE = {'pcam': 'CC0-1.0', 'crc': 'CC-BY-4.0', 'mhist': 'MHIST-RUA', 'bracs': 'CC-BY-NC-4.0',
                   'lunghist700': 'CC-BY-4.0', 'spider_breast': 'CC-BY-NC-4.0', 'camelyon17': 'CC0-1.0'}
MODEL_LICENSE = {'clip': 'MIT', 'plip': 'unspecified (research use)', 'conch': 'CC-BY-NC-ND-4.0', 'uni': 'CC-BY-NC-ND-4.0',
                 'virchow2': 'CC-BY-NC-ND-4.0', 'gigapath': 'Apache-2.0 (research use notice)',
                 'h_optimus_0': 'Apache-2.0', 'midnight': 'MIT'}
ALIASES = {'hoptimus': 'h_optimus_0', 'h-optimus-0': 'h_optimus_0', 'hoptimus0': 'h_optimus_0',
           'spider-breast': 'spider_breast', 'spider': 'spider_breast', 'lunghist': 'lunghist700'}
STD_FILES = {f'{s}_{k}.npy' for s in ('train', 'val', 'test') for k in ('features', 'labels')}

assert REPO_ID != 'YOUR_HF_USERNAME/PRISM-embeddings' or DRY_RUN, 'set REPO_ID'
assert os.path.isdir(REL_DIR), f'copy cam-ready/embeddings_release to {REL_DIR}'
sys.path.insert(0, f'{REL_DIR}/code')
from extract_embeddings import fingerprint, sha256   # one definition shared with the verifier
print('ok')

In [ ]:
# ---------------- 1. inventory -> release decision ----------------
inv = pd.read_csv(f'{ROOT}/embedding_inventory.csv')
assert set(['tree', 'model', 'group', 'file', 'shape', 'dtype', 'size_mb', 'sha256', 'path']) <= set(inv.columns)
canon = lambda x: ALIASES.get(str(x).strip().lower(), str(x).strip().lower())

rows = []
for r in inv.itertuples(index=False):
    m = canon(r.model)
    if r.tree == 'camelyon17':
        ds, hosp = 'camelyon17', r.group
        mh = re.fullmatch(r'hospital_(\d)', str(r.group))
        hosp = int(mh.group(1)) if mh else None
        hf_path = f'embeddings_camelyon17/{m}/{r.group}/{r.file}'
    else:
        ds, hosp = canon(r.group), None
        hf_path = f'embeddings/{m}/{ds}/{r.file}'
    split, kind = (r.file[:-4].split('_', 1) + [''])[:2] if r.file in STD_FILES else ('', '')
    reason = ''
    if r.file not in STD_FILES or m not in MODELS or (ds not in DATASETS and ds != 'camelyon17') \
            or (ds == 'camelyon17' and hosp is None):
        reason = 'not part of the PRISM release set (unexpected file or directory)'
    elif m not in RELEASE_MODELS:
        reason = WITHHELD_MODEL_REASON[m]
    elif ds not in RELEASE_DATASETS:
        reason = WITHHELD_DATASET_REASON[ds]
    rows.append(dict(hf_path=hf_path, tree=r.tree, model=m, dataset=ds, hospital=hosp, split=split, kind=kind,
                     file=r.file, shape=r.shape, dtype=r.dtype, size_mb=r.size_mb, sha256=r.sha256,
                     released=reason == '', withheld_reason=reason,
                     dataset_license=DATASET_LICENSE.get(ds, ''), model_license=MODEL_LICENSE.get(m, ''),
                     source_path=r.path))
ck = pd.DataFrame(rows)
assert not ck.hf_path.duplicated().any(), ck[ck.hf_path.duplicated(keep=False)]

# completeness: 48 main cells x 6 files and 40 Camelyon17 cells x 6 files are expected
ck['_h'] = ck.hospital.fillna(-1).astype(int)
have = ck[~ck.withheld_reason.str.startswith('not part')].groupby(['model', 'dataset', '_h']).size().to_dict()
exp = [(m, d, -1) for m in MODELS for d in DATASETS] + [(m, 'camelyon17', h) for m in MODELS for h in range(5)]
missing = [c for c in exp if have.get(c, 0) != 6]
print(f'inventory files: {len(ck)}  released: {ck.released.sum()}  withheld: {(~ck.released).sum()}')
print(f'released GB: {ck[ck.released].size_mb.sum() / 1024:.1f}   withheld GB: {ck[~ck.released].size_mb.sum() / 1024:.1f}')
print('cells with != 6 files:', missing or 'none')
print(ck[ck.tree == 'main'].pivot_table(index='model', columns='dataset', values='released',
      aggfunc=lambda s: 'R' if s.all() else ('-' if not s.any() else 'mixed')).reindex(MODELS).to_string())
print(ck[ck.tree == 'camelyon17'].groupby('model').released.mean().reindex(MODELS).map({1.0: 'R', 0.0: '-'}).to_string())
odd = ck[ck.withheld_reason.str.startswith('not part')]
if len(odd): print('\nunexpected files (listed in checksums.csv, not uploaded):\n', odd[['source_path']].to_string())

### 3b (optional, recommended). Split-index files for the main datasets
Needs the raw datasets under `MyDrive/PRISM/datasets`. The script reads file lists and metadata only, never images,
and stops if a rebuilt label vector differs from the stored `*_labels.npy`.

In [ ]:
RUN_SPLITS = True
if RUN_SPLITS:
    !pip -q install h5py
    !cd "{REL_DIR}/code" && python make_split_indices.py --datasets-root "{ROOT}/datasets" --emb-root "{ROOT}/embeddings" --out "{REL_DIR}/splits"

In [ ]:
# ---------------- 2. re-verify SHA256 + 3. fingerprints ----------------
t0 = time.time()
abs_path = lambda p: os.path.join(ROOT, p)
bad = []
for i, r in ck.iterrows():
    if r.withheld_reason.startswith('not part'):
        continue
    if r.released or REVERIFY_WITHHELD:
        h = sha256(abs_path(r.source_path))
        ck.at[i, 'sha256_verified'] = h == r.sha256
        if h != r.sha256:
            bad.append(r.source_path)
    if COMPUTE_STATS and r.kind == 'features':
        fp = fingerprint(abs_path(r.source_path))
        ck.at[i, 'mean_l2_norm'] = round(float(fp['mean_l2_norm']), 6)
        ck.at[i, 'global_mean'] = float(fp['global_mean'])
        ck.at[i, 'global_std'] = float(fp['global_std'])
        ck.at[i, 'colmean_proj8'] = ';'.join(f'{v:.8g}' for v in fp['colmean_proj8'])
    if i % 50 == 0:
        print(f'{i}/{len(ck)}  {time.time() - t0:.0f}s')
assert not bad, f'SHA256 mismatch against inventory (re-run 01_embedding_inventory or investigate): {bad}'
print('SHA256 verified for all released files', f'({time.time() - t0:.0f}s)')
if COMPUTE_STATS:
    print(ck[ck.kind == 'features'].groupby('model').mean_l2_norm.agg(['min', 'max']).round(3).to_string())

In [ ]:
# ---------------- 4. staging folder ----------------
if os.path.isdir(STAGE):
    for name in os.listdir(STAGE):          # keep .cache (upload_large_folder resume state) and parquet already built
        if name not in ('.cache', 'data'):
            p = os.path.join(STAGE, name)
            shutil.rmtree(p) if os.path.isdir(p) and not os.path.islink(p) else os.remove(p)
os.makedirs(STAGE, exist_ok=True)

rel = ck[ck.released]
for r in rel.itertuples():
    dst = os.path.join(STAGE, r.hf_path)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    os.symlink(abs_path(r.source_path), dst)

for sub in ('code', 'splits'):
    if os.path.isdir(f'{REL_DIR}/{sub}'):
        shutil.copytree(f'{REL_DIR}/{sub}', f'{STAGE}/{sub}', ignore=shutil.ignore_patterns('__pycache__', '*.pyc', '.DS_Store'))
# Camelyon17 index maps: repo copy, cross-checked with the Drive copy written by the extraction notebook
for fn in ('splits.json', 'resplit.json'):
    a, b = f'{STAGE}/splits/camelyon17/{fn}', f'{ROOT}/embeddings_camelyon17/{fn}'
    if os.path.exists(b):
        if os.path.exists(a):
            assert json.load(open(a)) == json.load(open(b)), f'{fn}: repo copy differs from Drive copy'
        else:
            os.makedirs(os.path.dirname(a), exist_ok=True); shutil.copy(b, a)
shutil.copy(f'{REL_DIR}/RELEASE_POLICY.md', f'{STAGE}/RELEASE_POLICY.md')

cols = ['hf_path', 'tree', 'model', 'dataset', 'hospital', 'split', 'kind', 'file', 'shape', 'dtype', 'size_mb',
        'sha256', 'released', 'withheld_reason', 'dataset_license', 'model_license',
        'mean_l2_norm', 'global_mean', 'global_std', 'colmean_proj8']
for c in cols:
    if c not in ck: ck[c] = ''
ck[cols].sort_values('hf_path').to_csv(f'{STAGE}/checksums.csv', index=False)
ck.to_csv(f'{ROOT}/embedding_checksums_full.csv', index=False)   # private copy incl. Drive source paths

# dataset card: inventory table + repo id
def shape_of(s): return tuple(int(x) for x in re.findall(r'\d+', s))
feat = rel[rel.kind == 'features'].copy()
feat['n'] = feat['shape'].map(lambda s: shape_of(s)[0]); feat['dim'] = feat['shape'].map(lambda s: shape_of(s)[1])
tab = feat.pivot_table(index=['dataset', 'model'], columns='split', values='n', aggfunc='sum').fillna(0).astype(int)
tab['dim'] = feat.groupby(['dataset', 'model']).dim.agg(lambda s: '/'.join(map(str, sorted(set(s)))))
lines = ['| dataset | model | train | val | test | dim |', '|---|---|---|---|---|---|']
for (d, m), r in tab.iterrows():
    lines.append(f"| {d} | {m} | {r.get('train', 0):,} | {r.get('val', 0):,} | {r.get('test', 0):,} | {r['dim']} |")
note = ('Rows per split and embedding dimension of the released `.npy` files, taken from `embedding_inventory.csv` '
        '(Camelyon17: summed over the 5 hospitals, stored split before re-splitting).')
card = open(f'{REL_DIR}/README.md').read()
card = re.sub(r'<!-- INVENTORY_TABLE_START -->.*?<!-- INVENTORY_TABLE_END -->',
              '<!-- INVENTORY_TABLE_START -->\n' + note + '\n\n' + '\n'.join(lines) + '\n<!-- INVENTORY_TABLE_END -->',
              card, flags=re.S)
card = card.replace('<HF_REPO_ID>', REPO_ID)
if not MAKE_PARQUET:   # drop the configs block: the .npy files alone do not form a loadable config
    card = re.sub(r'\nconfigs:\n(?:[ -].*\n)+', '\n', card)
open(f'{STAGE}/README.md', 'w').write(card)
print('\n'.join(lines))
print(f'\nstaged {len(rel)} .npy symlinks; checksums.csv rows: {len(ck)}')

In [ ]:
# ---------------- parquet copies for the datasets configs ----------------
import pyarrow as pa, pyarrow.parquet as pq
SPLIT_NAME = {'train': 'train', 'val': 'validation', 'test': 'test'}
CHUNK = 20_000

def to_table(model, dataset, rows_idx, y, X, extra=None):
    X = np.ascontiguousarray(X, dtype=np.float32); n, d = X.shape
    cols = {'model': pa.array([model] * n), 'dataset': pa.array([dataset] * n),
            'row': pa.array(np.asarray(rows_idx, dtype=np.int64)), 'label': pa.array(np.asarray(y, dtype=np.int64)),
            'embedding': pa.ListArray.from_arrays(pa.array(np.arange(0, (n + 1) * d, d, dtype=np.int32)),
                                                  pa.array(X.ravel()))}
    for k, v in (extra or {}).items(): cols[k] = v
    return pa.table(cols)

def write_split(path, batches):
    tmp = path + '.tmp'
    os.makedirs(os.path.dirname(path), exist_ok=True)
    w = None
    for t in batches:
        w = w or pq.ParquetWriter(tmp, t.schema, compression='zstd')
        w.write_table(t, row_group_size=CHUNK)
    w.close(); os.replace(tmp, path)

if MAKE_PARQUET:
    t0 = time.time()
    main = rel[(rel.tree == 'main') & (rel.kind == 'features')]
    for r in main.itertuples():
        out = f'{STAGE}/data/{r.dataset}/{r.model}/{SPLIT_NAME[r.split]}.parquet'
        if os.path.exists(out): continue
        X = np.load(abs_path(r.source_path), mmap_mode='r')
        y = np.load(abs_path(r.source_path.replace('_features.npy', '_labels.npy')))
        assert len(X) == len(y)
        write_split(out, (to_table(r.model, r.dataset, np.arange(i, min(i + CHUNK, len(X))), y[i:i + CHUNK], X[i:i + CHUNK])
                          for i in range(0, len(X), CHUNK)))
        print(f'{out[len(STAGE)+1:]}  {time.time() - t0:.0f}s')

    # Camelyon17: write in the paper's balanced re-split (resplit.json indexes concat(train, val, test) per hospital)
    resplit = json.load(open(f'{STAGE}/splits/camelyon17/resplit.json'))
    cam = rel[(rel.tree == 'camelyon17') & (rel.kind == 'features')]
    for m in sorted(cam.model.unique()):
        for s in ('train', 'val', 'test'):
            out = f'{STAGE}/data/camelyon17/{m}/{SPLIT_NAME[s]}.parquet'
            if os.path.exists(out): continue
            def batches():
                for h in range(5):
                    base = f'{ROOT}/' + os.path.dirname(cam[(cam.model == m) & (cam.hospital == h)].source_path.iloc[0])
                    Xs = [np.load(f'{base}/{p}_features.npy', mmap_mode='r') for p in ('train', 'val', 'test')]
                    ys = [np.load(f'{base}/{p}_labels.npy') for p in ('train', 'val', 'test')]
                    off = np.cumsum([0] + [len(x) for x in Xs])
                    pos = np.asarray(resplit[str(h)][s], dtype=np.int64)
                    assert pos.max() < off[-1], f'resplit index out of range for {m} hospital {h}'
                    src = np.searchsorted(off, pos, side='right') - 1
                    X = np.vstack([np.asarray(Xs[k][pos[src == k] - off[k]]) for k in range(3)])
                    y = np.concatenate([ys[k][pos[src == k] - off[k]] for k in range(3)])
                    p_sorted = np.concatenate([pos[src == k] for k in range(3)])
                    osplit = np.concatenate([[('train', 'val', 'test')[k]] * int((src == k).sum()) for k in range(3)])
                    assert abs(y.mean() - 0.5) < 0.01, f'{m} h{h} {s}: tumour rate {y.mean():.3f}'
                    for i in range(0, len(X), CHUNK):
                        yield to_table(m, 'camelyon17', p_sorted[i:i + CHUNK], y[i:i + CHUNK], X[i:i + CHUNK],
                                       {'hospital': pa.array(np.full(min(CHUNK, len(X) - i), h, dtype=np.int64)),
                                        'orig_split': pa.array(osplit[i:i + CHUNK].tolist())})
            write_split(out, batches())
            print(f'{out[len(STAGE)+1:]}  {time.time() - t0:.0f}s')
    print('parquet GB:', round(sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(f'{STAGE}/data')
                                   for f in fs) / 2**30, 2))

In [ ]:
# ---------------- 5. create private repo, gate, upload ----------------
staged = sorted(os.path.relpath(os.path.join(dp, f), STAGE) for dp, _, fs in os.walk(STAGE, followlinks=True)
                for f in fs if not os.path.relpath(dp, STAGE).startswith('.cache') and not f.endswith('.tmp'))
print(f'{len(staged)} files staged')
if DRY_RUN:
    print('DRY_RUN: nothing uploaded')
else:
    from huggingface_hub import HfApi
    api = HfApi(token=userdata.get('HF_TOKEN'))
    print('user:', api.whoami()['name'])
    api.create_repo(REPO_ID, repo_type='dataset', private=True, exist_ok=True)
    if api.repo_info(REPO_ID, repo_type='dataset').private is False:
        print('WARNING: the repo already exists and is PUBLIC; files will be visible immediately.')
    if GATED:
        api.update_repo_settings(repo_id=REPO_ID, repo_type='dataset', gated=GATED)
    api.upload_large_folder(repo_id=REPO_ID, repo_type='dataset', folder_path=STAGE,
                            ignore_patterns=['*.tmp'], num_workers=8)

In [ ]:
# ---------------- 6. check the remote and print the URL ----------------
if not DRY_RUN:
    remote = set(api.list_repo_files(REPO_ID, repo_type='dataset'))
    missing_remote = [f for f in staged if f not in remote]
    withheld_remote = [f for f in remote if f in set(ck[~ck.released].hf_path)]
    info = api.repo_info(REPO_ID, repo_type='dataset')
    print(f'remote files: {len(remote)}  missing: {len(missing_remote)}  withheld files present (must be 0): {len(withheld_remote)}')
    assert not withheld_remote, withheld_remote
    if missing_remote: print('re-run the upload cell; missing e.g.', missing_remote[:5])
    print(f'private={info.private}  gated={getattr(info, "gated", None)}')
    print('URL:', f'https://huggingface.co/datasets/{REPO_ID}')

### 7. Make public (manual, after review)
Check on the Hub that the card renders, the configs load (`load_dataset(REPO_ID, "bracs")`), and `checksums.csv`
lists withheld files without any withheld `.npy` present. Then uncomment and run:

In [ ]:
# api.update_repo_settings(repo_id=REPO_ID, repo_type='dataset', private=False)
# print('public:', f'https://huggingface.co/datasets/{REPO_ID}')